In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": False, "run_validation": True, "run_network_validation": True, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit"], "activitysim-validation-notebook": ["work_from_home", "auto_ownership", "telecommute_frequency", "free_parking", "cdap", "intermediate_stop_frequency", "trip_purpose", "trip_destination_choice", "school_location", "work_location", "mandatory_tour_frequency", "mandatory_tour_scheduling", "non_mandatory_tour_frequency", "non_mandatory_tour_destination_choice", "non_mandatory_tour_scheduling", "joint_tour_frequency", "joint_tour_composition", "atwork_subtours_frequency", "atwork_subtours_destination_choice", "atwork_subtours_scheduling", "atwork_subtour_mode", "tour_mode_choice", "trip_mode_choice"], "daysim-validation-notebook": ["time_choice", "tours", "tour_destination", "transit_pass_ownership", "trips", "trip_destination", "workbased_subtour_generation", "workbased_subtour_mode", "work_location", "work_tour_mode", "work_trip_mode"], "network-validation-notebook": ["JBLM", "supplementals", "transit_validation", "traffic_validation", "bike_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "../../../../", "survey_directories": {"survey": "../../../../inputs/base_year/survey"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 5, "9to15": 9, "15to18": 15, "18to20": 18, "20to5": 20}, "summer_list": [87], "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "base_year_2023_final", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/rtp_2026_2050", "abm_model": "daysim", "run_accessibility_calcs": False, "run_setup_emme_project_folders": False, "run_setup_emme_bank_folders": False, "run_copy_scenario_inputs": False, "run_import_networks": False, "run_skims_and_paths_free_flow": False, "run_skims_and_paths": False, "run_truck_model": False, "run_supplemental_trips": False, "run_daysim": False, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_delivery": False, "include_telecommute": True, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": True, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
import plotly.express as px
import polars as pl
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [3]:
tour = util.get_tour_data(summary_config)
trip = util.get_trip_data(summary_config)


In [4]:
df_tour = tour.clone()
df_trip = trip.clone()

purpose_order = ["Work", "School", "Shop", "Meal", "Escort", "Personal Business", "Social"]

pdpurp_cat = {1: "Work", 2: "School", 3: "Escort", 4: "Personal Business",
              5: "Shop", 6: "Meal", 7: "Social", 8: "Social", 9: "Personal Business"}
dpurp_cat = pdpurp_cat

df_tour = df_tour.with_columns([
    pl.col('pdpurp').replace_strict(pdpurp_cat, default=None).alias('pdpurp_label'),
    pl.col('tlvorig').truediv(60).floor().cast(pl.Int64).alias('tlvorg_hr')
]).with_columns(
    pl.col('pdpurp_label').cast(pl.Enum(purpose_order))
)

df_trip = df_trip.with_columns([
    pl.col('dpurp').replace_strict(dpurp_cat, default=None).alias('dpurp_label'),
    pl.col('deptm').truediv(60).floor().cast(pl.Int64).alias('dept_hr')
]).with_columns(
    pl.col('dpurp_label').cast(pl.Enum(purpose_order))
)


## departure hour

In [5]:
df_plot = (
    df_tour.group_by(['tlvorg_hr', 'source'])
    .agg([
        pl.col('toexpfac').count().alias('sample_count'),
        pl.col('toexpfac').sum().alias('toexpfac')
    ])
    .with_columns((pl.col('toexpfac') / pl.col('toexpfac').sum().over(['source'])).alias('percentage'))
)

fig = px.bar(df_plot.to_pandas(), x="tlvorg_hr", y="percentage", color="source",
             barmode="group", title="tour origin departure hour",
             hover_data=['toexpfac', 'sample_count'])
fig.update_layout(height=400, width=700, font=dict(size=11),
                  xaxis=dict(dtick=1, categoryorder='category ascending'),
                  yaxis=dict(tickformat=".0%"))
fig.show()

In [6]:
df_plot = (
    df_trip.group_by(['dept_hr', 'source'])
    .agg([
        pl.col('trexpfac').count().alias('sample_count'),
        pl.col('trexpfac').sum().alias('trexpfac')
    ])
    .with_columns((pl.col('trexpfac') / pl.col('trexpfac').sum().over(['source'])).alias('percentage'))
)

fig = px.bar(df_plot.to_pandas(), x="dept_hr", y="percentage", color="source",
             barmode="group", title="trip origin departure hour",
             hover_data=['trexpfac', 'sample_count'])
fig.update_layout(height=400, width=700, font=dict(size=11),
                  xaxis=dict(dtick=1, categoryorder='category ascending'),
                  yaxis=dict(tickformat=".0%"))
fig.show()

## time choice by purpose

In [7]:
df_plot = (
    df_tour.group_by(['tlvorg_hr', 'pdpurp_label', 'source'])
    .agg([
        pl.col('toexpfac').count().alias('sample_count'),
        pl.col('toexpfac').sum().alias('toexpfac')
    ])
    .with_columns(
        (pl.col('toexpfac') / pl.col('toexpfac').sum().over(['pdpurp_label', 'source'])).alias('percentage')
    )
)

fig = px.bar(df_plot.sort(['source', 'pdpurp_label']).to_pandas(),
             x="tlvorg_hr", y="percentage", color="source", barmode="group",
             facet_row='pdpurp_label',
             hover_data=['toexpfac', 'sample_count'],
             title="tour origin departure hour by purpose")
fig.update_layout(height=1500, width=850)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.for_each_xaxis(lambda a: a.update(dict(dtick=1, categoryorder='category ascending')))
fig.for_each_yaxis(lambda a: a.update(dict(tickformat=".0%")))
fig.show()

In [8]:
df_plot = (
    df_trip.group_by(['dept_hr', 'dpurp_label', 'source'])
    .agg([
        pl.col('trexpfac').count().alias('sample_count'),
        pl.col('trexpfac').sum().alias('trexpfac')
    ])
    .with_columns(
        (pl.col('trexpfac') / pl.col('trexpfac').sum().over(['dpurp_label', 'source'])).alias('percentage')
    )
)

fig = px.bar(df_plot.sort(['source', 'dpurp_label']).to_pandas(),
             x="dept_hr", y="percentage", color="source", barmode="group",
             facet_row='dpurp_label',
             hover_data=['trexpfac', 'sample_count'],
             title="trip origin departure hour by purpose")
fig.update_layout(height=1500, width=850)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.for_each_xaxis(lambda a: a.update(dict(dtick=1, categoryorder='category ascending')))
fig.for_each_yaxis(lambda a: a.update(dict(tickformat=".0%")))
fig.show()